# Ella2027 — Isolated Real-ESRGAN Upscaler Test

دفتر اختبار مستقل. لا يغيّر دفتر WanGP ولا يوقف العامل. ينشئ Python venv منفصلاً مع system-site-packages ليستعمل Torch/Torchvision الموجودين دون إعادة تثبيتهما. الاختبار يعالج إطاراً واحداً فقط من أحدث فيديو، باستخدام GPU 1، ثم يحفظ صورة 1280×720 للتحقق من الجودة والذاكرة.

**قبل التشغيل:** تأكد أن العامل لا يعالج فيديو جديداً الآن. لا تشغّل هذا الاختبار بالتزامن مع توليد فيديو.

In [ ]:
import os, sys, glob, json, shutil, subprocess, textwrap
from pathlib import Path

VIDEO_DIR = Path('/kaggle/working/ella2027-output')
videos = sorted([p for p in VIDEO_DIR.glob('*') if p.is_file() and p.suffix.lower() in {'.mp4','.mov','.mkv','.webm'}], key=lambda p:p.stat().st_mtime, reverse=True)
if not videos:
    raise FileNotFoundError(f'No generated video found in {VIDEO_DIR}. Generate one first, then rerun this test.')
VIDEO = videos[0]
print('Test video:', VIDEO)
print('Size:', round(VIDEO.stat().st_size/1024/1024, 2), 'MiB')
print('Torch in main runtime:', __import__('torch').__version__)
print('Torchvision in main runtime:', __import__('torchvision').__version__)

VENV = Path('/kaggle/working/ella-realesrgan-venv')
PY = VENV / 'bin' / 'python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
print('Isolated environment:', VENV)

# Install only the upscaler packages inside the venv. --no-deps prevents pip from replacing Torch/CUDA.
subprocess.run([str(PY), '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', 'basicsr==1.4.2', 'realesrgan==0.3.0'], check=True)
site = subprocess.check_output([str(PY), '-c', 'import site; print(site.getsitepackages()[0])'], text=True).strip()
shim = '''import sys, types\ntry:\n    from torchvision.transforms import functional as _F\n    _m = types.ModuleType("torchvision.transforms.functional_tensor")\n    _m.rgb_to_grayscale = _F.rgb_to_grayscale\n    sys.modules.setdefault("torchvision.transforms.functional_tensor", _m)\nexcept Exception as _e:\n    print("[ELLA-UPSCALER] torchvision compatibility shim unavailable:", _e)\n'''
Path(site, 'sitecustomize.py').write_text(shim, encoding='utf-8')

# Add only lightweight BasicSR runtime dependencies to the isolated venv.
subprocess.run([str(PY), '-m', 'pip', 'install', '--no-cache-dir', 'addict', 'future', 'lmdb', 'PyYAML', 'scipy', 'tqdm', 'yapf', 'tensorboard'], check=True)
check = '''import torch, torchvision, cv2\nfrom basicsr.archs.rrdbnet_arch import RRDBNet\nfrom realesrgan import RealESRGANer\nprint("IMPORT_OK", torch.__version__, torchvision.__version__, torch.cuda.is_available(), cv2.__version__)\n'''
subprocess.run([str(PY), '-c', check], check=True)
print('Isolated Real-ESRGAN imports passed; main WanGP Python environment was not modified.')

# Download the official RealESRGAN_x4plus weights if absent.
WEIGHTS = Path('/kaggle/working/RealESRGAN_x4plus.pth')
if not WEIGHTS.exists() or WEIGHTS.stat().st_size < 50_000_000:
    import requests
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth'
    print('Downloading official RealESRGAN_x4plus weights...')
    r = requests.get(url, stream=True, timeout=(30, 300)); r.raise_for_status()
    tmp = WEIGHTS.with_suffix('.pth.part')
    with tmp.open('wb') as f:
        for chunk in r.iter_content(1024*1024):
            if chunk: f.write(chunk)
    tmp.replace(WEIGHTS)
print('Weights:', round(WEIGHTS.stat().st_size/1024/1024, 1), 'MiB')

OUT = Path('/kaggle/working/ella2027-upscaler-test')
OUT.mkdir(parents=True, exist_ok=True)
result_script = r'''import sys, cv2, torch
from pathlib import Path
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
video, weights, out = map(str, sys.argv[1:4])
cap = cv2.VideoCapture(video)
ok, frame = cap.read(); cap.release()
if not ok: raise RuntimeError('Could not decode the first video frame.')
h, w = frame.shape[:2]
print('INPUT_FRAME', w, h, flush=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda': raise RuntimeError('CUDA is unavailable in the isolated upscaler process.')
torch.cuda.set_device(0)
model = RRDBNet(num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32)
upsampler = RealESRGANer(scale=4, model_path=weights, model=model, tile=256, tile_pad=16, pre_pad=0, half=True, device=device)
output, _ = upsampler.enhance(frame, outscale=1.25)
oh, ow = output.shape[:2]
if (ow, oh) != (1280, 720): raise RuntimeError(f'Unexpected output dimensions: {ow}x{oh}')
if not cv2.imwrite(out, output): raise RuntimeError('Failed to save upscaled frame.')
print('UPSCALE_OK', ow, oh, 'GPU', torch.cuda.get_device_name(0), 'OUTPUT', out, flush=True)
'''
script_path = OUT / 'upscale_one_frame.py'
script_path.write_text(result_script, encoding='utf-8')
env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = '1'
subprocess.run([str(PY), str(script_path), str(VIDEO), str(WEIGHTS), str(OUT/'frame_1280x720.png')], check=True, env=env)

img = __import__('PIL.Image', fromlist=['Image']).open(OUT/'frame_1280x720.png')
print('\n=== TEST PASSED ===')
print('Upscaled frame:', OUT/'frame_1280x720.png')
print('Dimensions:', img.size)
print('Original video untouched. Worker process untouched.')

## What this test proves

نجاح هذا الاختبار يثبت أن تثبيت Real-ESRGAN المعزول متوافق وأنه يستطيع معالجة إطار واحد إلى 1280×720. لا يعني ذلك بعدُ أن معالجة الفيديو كاملاً خالية من الوميض أو أنها أسرع من الزمن المقبول. الخطوة التالية بعد فحص الصورة هي اختبار مقطع قصير مع الصوت الأصلي والتحقق بـ ffprobe.